In [1]:
from skidl.logger import stop_log_file_output
stop_log_file_output(True)

In [2]:
from python.spice_tools import search_spice_model, save_part_model

entry = search_spice_model(name="VBUS05L1-DD1", library="TVS_Fuse_Board_Level_Protection")
if entry:
    print(entry["model_content"])
else:
    print("Not in model db")
 

* VBUS05L1-DD1 - Ultra Low Capacitance Bidirectional TVS Diode
* Manufacturer: Leiditech
* Package: DFN1006, 2 pins (bidirectional between pins 1 and 2)
*
* Datasheet key specs (TA = 25°C unless noted):
*   - Reverse working voltage VRWM = 5 V
*   - Breakdown voltage VBR = 6 V at IT = 1 mA
*   - Reverse leakage current IR(max) = 0.5 µA at VRWM = 5 V
*   - Clamping voltage VC ≈ 12 V at IPP = 1 A (8/20 µs pulse)
*   - Clamping voltage VC ≈ 23 V at IPP = 4.5 A (8/20 µs pulse)
*   - Junction capacitance CJ(typ) = 0.22 pF at VR = 0 V, f = 1 MHz
*   - Peak pulse power (8/20 µs) Ppp = 110 W (not explicitly enforced in model)
*
* Behavioral modeling notes:
*   - Implemented as two identical avalanche diodes in series and opposite polarity
*     to emulate a bidirectional TVS centered at 0 V.
*   - Series resistance of each diode is chosen so that the pair clamps near 23 V
*     at 4.5 A (8/20 µs), matching the higher-current clamping spec; this makes the
*     1 A clamping voltage somewhat low

In [3]:
#    # Save part model
# from python.spice_tools import save_part_model

# with open("/root/workspace/KiCAD_MCP/test_cases/test_charger_3A/spice/usb_power_in/SH2_U262_161N_4BVC11.spice.lib", "r") as f:
#     model_content = f.read()
# save_part_model(
# name="SH2-U262-161N-4BVC11",
# library="Uncategorized",
# model_content=model_content,
# vendor_provided=False,
# )

In [4]:
from pathlib import Path
from python.spice_tools import convert_skidl_module

name = convert_skidl_module(
    input_path=Path("/root/workspace/KiCAD_MCP/test_cases/test_charger_3A/skidl/modules/usb_power_in.py"),
    subckt_name="USB_POWER_IN",
    output_path=Path("test_cases/test_charger_3A/spice/usb_power_in/dut.py"),
)

name

'USB_POWER_IN_pyspice'

In [5]:
import json

test_bench_path = "test_cases/test_charger_3A/testbench/usb_power_in_testbench.json"

with open(test_bench_path, "r") as f:
    test_bench = json.load(f)
    testcases = test_bench["use_cases"]
case_ids = [case["name"] for case in testcases]

case_ids

['steady_state_3a_nominal_vbus',
 'steady_state_3p5a_max_vbus',
 'usb_attach_ramp_inrush_vbus']

In [6]:
from python.spice_tools.harness_sanity import harness_sanity_check

for idx in range(len(case_ids)):
    harness_path = f"test_cases/test_charger_3A/spice/usb_power_in/testcases/{case_ids[idx]}.py"

    result = harness_sanity_check(harness_path=harness_path)
    print(result)


{'ok': True, 'max_abs_voltage': 4.91, 'max_abs_current': 0.0, 'num_points': 508, 'error': None}
{'ok': True, 'max_abs_voltage': 3.5, 'max_abs_current': 0.0, 'num_points': 508, 'error': None}
{'ok': True, 'max_abs_voltage': 4.995000000000001, 'max_abs_current': 0.0, 'num_points': 511, 'error': None}


In [7]:
from python.spice_tools.testbench_runner import run_use_case

for idx in range(len(case_ids)):
    harness_path = f"test_cases/test_charger_3A/spice/usb_power_in/testcases/{case_ids[idx]}.py"

    reports = run_use_case(
        schema_path=test_bench_path,
        harness_path=harness_path,
        use_case_name=case_ids[idx],
        dut_path="test_cases/test_charger_3A/spice/usb_power_in/dut.py",
        dut_module_name="USB_POWER_IN_pyspice",
    )

    print(reports)


{'steady_state_3a_nominal_vbus': {'total_measurements': 2, 'num_passed': 0, 'all_passed': False, 'measurements': {'vbus_mean_3a_nominal': {'assertion': {'value': 7.815829852829253e-25, 'op': '>=', 'limit': 4.7}, 'passed': False}, 'vbus_inband_3a_nominal': {'assertion': {'value': 0.0, 'op': '>=', 'limit': 4.5}, 'passed': False}}}}
{'steady_state_3p5a_max_vbus': {'total_measurements': 2, 'num_passed': 0, 'all_passed': False, 'measurements': {'vbus_mean_3p5a_max': {'assertion': {'value': 7.815829852829253e-25, 'op': '>=', 'limit': 4.6}, 'passed': False}, 'vbus_inband_3p5a_max': {'assertion': {'value': 0.0, 'op': '>=', 'limit': 4.5}, 'passed': False}}}}
{'usb_attach_ramp_inrush_vbus': {'total_measurements': 2, 'num_passed': 1, 'all_passed': False, 'measurements': {'vbus_time_to_4p5v_attach': {'assertion': {'value': 1e+308, 'op': '<=', 'limit': 2.0}, 'passed': False}, 'vbus_overshoot_attach': {'assertion': {'value': 0.0, 'op': '<=', 'limit': 0.5}, 'passed': True}}}}


In [ ]:
import json, sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd() / "python"))
from commands.database_tools.library_schematic import LibraryManager
from python.spice_tools.utils import validate_spice_model

mpn = "SH2-U262-161N-4BVC11"
lib = "Uncategorized"

result = LibraryManager.get_symbol_pinout({
    "library": lib,
    "symbol": mpn,
})

pins = result.get("pins")
print(pins)

res = validate_spice_model(
    model_path="test_cases/test_charger_3A/spice/usb_power_in/SH2_U262_161N_4BVC11.spice.lib",
    expected_subckt_name=mpn,
    expected_pinout=pins,
)

In [ ]:
for r in res:
    print(r+'\n\n')